In [2]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import hashlib
import json

import pandas as pd
import yaml
import yfinance as yf

from IPython.display import display


ROOT = Path.cwd().resolve()

if not (ROOT / "configs" / "base.yaml").exists():
    ROOT = ROOT.parent

if not (ROOT / "configs" / "base.yaml").exists():
    raise FileNotFoundError("Open the notebook from the project directory.")

RAW_DIR = ROOT / "data" / "raw" / "development"
RESULTS_DIR = ROOT / "results"

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)

/Users/puppixx1610/Projects/pairs-trading-research


In [3]:
with (ROOT / "configs" / "base.yaml").open() as file:
    config = yaml.safe_load(file)

groups = config["groups"]
tickers = sorted(
    ticker
    for members in groups.values()
    for ticker in members
)

start = config["data"]["start"]
end = config["data"]["end_exclusive"]
interval = config["data"]["interval"]

assert len(tickers) == len(set(tickers)) == 12
assert start == "2010-01-01"
assert end == "2020-01-01"

universe = pd.DataFrame(
    [
        {"group": group, "ticker": ticker}
        for group, members in groups.items()
        for ticker in members
    ]
)

display(universe)

,group,ticker
0,us_large_cap,SPY
1,us_large_cap,IVV
2,us_large_cap,VOO
3,us_total_market,VTI
4,us_total_market,ITOT
5,us_total_market,SCHB
6,developed_ex_us,EFA
7,developed_ex_us,VEA
8,developed_ex_us,SCHF
9,emerging_markets,EEM


In [4]:
settings = {
    "start": start,
    "end": end,
    "interval": interval,
    "auto_adjust": False,
    "actions": True,
    "keepna": True,
    "repair": False,
}

required_columns = {
    "Open", "High", "Low", "Close", "Adj Close", "Volume"
}

for ticker in tickers:
    csv_path = RAW_DIR / f"{ticker}.csv"
    metadata_path = RAW_DIR / f"{ticker}.metadata.json"

    if csv_path.exists():
        if not metadata_path.exists():
            raise FileNotFoundError(f"{ticker}: metadata file is missing.")

        metadata = json.loads(metadata_path.read_text())

        saved_settings = metadata.get("download_settings", metadata)
        saved_end = saved_settings.get(
            "end", saved_settings.get("end_exclusive")
        )

        if (
            saved_settings.get("start") != start
            or saved_end != end
            or saved_settings.get("interval") != interval
            or saved_settings.get("auto_adjust") is not False
        ):
            raise ValueError(f"{ticker}: cached download settings differ.")

        checksum = hashlib.sha256(csv_path.read_bytes()).hexdigest()

        if checksum != metadata.get("sha256"):
            raise ValueError(f"{ticker}: cached file checksum differs.")

        print(f"{ticker}: using saved data")
        continue

    if metadata_path.exists():
        raise FileExistsError(f"{ticker}: metadata exists without a CSV.")

    data = yf.download(
        ticker,
        **settings,
        threads=False,
        progress=False,
        multi_level_index=False,
    )

    if data is None or data.empty:
        raise ValueError(f"{ticker}: download returned no data.")

    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise ValueError(f"{ticker}: missing columns {missing_columns}")

    data.to_csv(csv_path, index_label="Date")

    metadata = {
        "ticker": ticker,
        "source": "Yahoo Finance",
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "yfinance_version": yf.__version__,
        "download_settings": settings,
        "sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    }

    metadata_path.write_text(json.dumps(metadata, indent=2))

    print(f"{ticker}: saved {len(data):,} rows")

/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

EEM: saved 2,516 rows


/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

EFA: saved 2,516 rows


/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

...

/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

VOO: saved 2,344 rows


/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

VTI: saved 2,516 rows
VWO: saved 2,516 rows


/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/puppixx1610/Projects/pairs-trading-research/.venv/lib/python3.14/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.or

In [5]:
raw = {}
coverage_rows = []

for group, members in groups.items():
    for ticker in members:
        data = pd.read_csv(
            RAW_DIR / f"{ticker}.csv",
            index_col="Date",
            parse_dates=True,
        )

        data.index = pd.DatetimeIndex(data.index)

        if data.index.tz is not None:
            data.index = data.index.tz_localize(None)

        data.index = data.index.normalize()

        assert not data.empty, f"{ticker}: empty dataset"
        assert required_columns.issubset(data.columns), ticker
        assert not data.index.hasnans, f"{ticker}: invalid dates"
        assert not data.index.duplicated().any(), ticker
        assert data.index.is_monotonic_increasing, ticker
        assert data.index.min() >= pd.Timestamp(start), ticker
        assert data.index.max() < pd.Timestamp(end), ticker

        valid_prices = data["Adj Close"].dropna()

        assert not valid_prices.empty, f"{ticker}: no adjusted prices"

        raw[ticker] = data

        coverage_rows.append({
            "group": group,
            "ticker": ticker,
            "rows": len(data),
            "valid_prices": len(valid_prices),
            "first_valid": valid_prices.index.min(),
            "last_valid": valid_prices.index.max(),
            "missing_prices": data["Adj Close"].isna().sum(),
            "dividend_data": "Dividends" in data.columns,
            "split_data": "Stock Splits" in data.columns,
        })

coverage = pd.DataFrame(coverage_rows)

coverage.to_csv(
    RESULTS_DIR / "download_coverage_development.csv",
    index=False,
)

display(coverage)

,group,ticker,rows,valid_prices,first_valid,last_valid,missing_prices,dividend_data,split_data
0,us_large_cap,SPY,2516,2516,2010-01-04,2019-12-31,0,True,True
1,us_large_cap,IVV,2516,2516,2010-01-04,2019-12-31,0,True,True
2,us_large_cap,VOO,2344,2344,2010-09-09,2019-12-31,0,True,True
3,us_total_market,VTI,2516,2516,2010-01-04,2019-12-31,0,True,True
4,us_total_market,ITOT,2516,2516,2010-01-04,2019-12-31,0,True,True
5,us_total_market,SCHB,2516,2516,2010-01-04,2019-12-31,0,True,True
6,developed_ex_us,EFA,2516,2516,2010-01-04,2019-12-31,0,True,True
7,developed_ex_us,VEA,2516,2516,2010-01-04,2019-12-31,0,True,True
8,developed_ex_us,SCHF,2516,2516,2010-01-04,2019-12-31,0,True,True
9,emerging_markets,EEM,2516,2516,2010-01-04,2019-12-31,0,True,True


In [6]:
candidate_pairs = pd.DataFrame(
    [
        {"group": group, "A": a, "B": b}
        for group, members in groups.items()
        for a, b in combinations(sorted(members), 2)
    ]
)

assert len(candidate_pairs) == 12

candidate_pairs.to_csv(
    RESULTS_DIR / "candidate_pairs.csv",
    index=False,
)

display(candidate_pairs)

,group,A,B
0,us_large_cap,IVV,SPY
1,us_large_cap,IVV,VOO
2,us_large_cap,SPY,VOO
3,us_total_market,ITOT,SCHB
4,us_total_market,ITOT,VTI
5,us_total_market,SCHB,VTI
6,developed_ex_us,EFA,SCHF
7,developed_ex_us,EFA,VEA
8,developed_ex_us,SCHF,VEA
9,emerging_markets,EEM,IEMG


In [7]:
display(raw["SPY"].head())

display(
    coverage.loc[
        coverage["ticker"].isin(["VOO", "IEMG"]),
        ["ticker", "first_valid", "last_valid", "valid_prices"],
    ]
)

print(f"Loaded {len(raw)} ETFs and defined {len(candidate_pairs)} pairs.")

,Adj Close,Capital Gains,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,,
2010-01-04,84.368996,0.0,113.330002,0.0,113.389999,111.510002,112.370003,0.0,118944600
2010-01-05,84.592285,0.0,113.629997,0.0,113.680000,112.849998,113.260002,0.0,111579900
2010-01-06,84.651871,0.0,113.709999,0.0,113.989998,113.430000,113.519997,0.0,116074400
2010-01-07,85.009201,0.0,114.190002,0.0,114.330002,113.180000,113.500000,0.0,131091100
2010-01-08,85.292084,0.0,114.570000,0.0,114.620003,113.660004,113.889999,0.0,126402800


,ticker,first_valid,last_valid,valid_prices
2,VOO,2010-09-09,2019-12-31,2344
11,IEMG,2012-10-24,2019-12-31,1807


Loaded 12 ETFs and defined 12 pairs.
